# Suite MUT — Mutations: the schemas of a mutation log

`Mutations` holds the schemas of a mutation log ([Serialization](../../docs/SERIALIZATION.md), Mutations : Vocabulary):
reference verbs at the top level (`Log`, a list of `Reference`), value verbs nested in them, and the nouns `Ref`, `Def`
and `Slice`. Mutations are schema-governed: every store holds their schemas, and a log is built and written as any value
is. Every store also holds the basic scalars, `Schemas.OfBool` to `Schemas.OfBytes`.

In [ ]:
import json
from types import SimpleNamespace

from mbse.Schemas.Framework import Builders as B, JSON, Plain
from support import raises

store = B.Store()
M = SimpleNamespace(  # the store's own mutation schemas, through its paths
    Log=store.Mutations.Log.as_schema(), Reference=store.Mutations.Reference.as_schema(),
    OfAny=store.Mutations.OfAny.as_schema(), OfContainer=store.Mutations.OfContainer.as_schema(),
    OfLink=store.Mutations.OfLink.as_schema(), PropertySet=store.Mutations.OfProperty.Set.as_schema(),
    AnySet=store.Mutations.OfAny.Set.as_schema(), ScalarSet=store.Mutations.OfScalar.Set.as_schema(),
    SCHEMAS=[store.registered(name) for name in store.names() if name.startswith("Mutations.")])


def at(singleton, *names):
    """A path, a `Paths.OfRoot` value: from `singleton`, then, for a schema object's, through each module and then the
    schema by name (the root module is the singleton ".")."""
    def step(rest):
        if not rest:
            return lambda n: n.target(lambda t: t)
        return lambda n: n.entry(lambda e: e.i("schemas").o("").properties({"name": rest[0]}).next(step(rest[1:])))
    return store.Paths.OfRoot(lambda r: r.singleton(singleton).next(step(names)))


def written_at(singleton, *names):
    """`at(singleton, *names)` as plain data."""
    nested = {"target": {}}
    for name in reversed(names):
        nested = {"entry": {"i": "schemas", "o": "", "properties": {"name": name}, "next": nested}}
    return {"singleton": singleton, "next": nested}


def ref(singleton, *names):
    """A `Mutations.OfObject.Ref` naming the object at `at(singleton, *names)`."""
    return store.Mutations.OfObject.Ref(lambda r: r.path(at(singleton, *names)))


def log(*verbs):
    """A `Mutations.Log` of `verbs`, each a `Mutations.Reference` value."""
    return store.Mutations.Log(lambda c: [c.at(i).as_union(v) for i, v in enumerate(verbs)])


def plain(value, schema=M.Log):
    return Plain.ToPlain(store)(schema, value)

## MUT-01 · The mutation schemas, valid, and the basic scalars, held by every store

In [ ]:
# A mutation nests mutations, so the verbs hold themselves, which validation refuses of a value schema (Object identity,
# Decided, 72): an open question until 4f replaces the log. That is each one's only problem.
cycle = "a value schema cannot hold itself: value object schemas cannot have cycles; reference object schemas can"
assert all(problem.endswith(cycle) for schema in M.SCHEMAS for problem in schema.validate()) and len(M.SCHEMAS) == 42
for schema in M.SCHEMAS:
    assert store.registered(schema.name) is schema
other = B.Store()
assert other.registered("Mutations.Log") is not M.Log and all(p.endswith(cycle) for p in other.Mutations.Log.as_schema().validate())  # each its own
for name in ("OfBool", "OfInt", "OfFloat", "OfStr", "OfBytes"):
    scalar = getattr(store.Schemas, name).as_schema()
    assert scalar.name == f"Schemas.{name}" and store.registered(scalar.name) is scalar
assert store.Schemas.OfStr.as_schema().type is str and store.Schemas.OfObject.as_schema().properties["ref"].schema is store.Schemas.OfBool.as_schema()
assert [b.name for b in M.Reference.branches] == ["exists", "as_property", "insert", "remove", "clear", "as_entries"]
assert [b.name for b in M.OfAny.branches] == ["set", "clear", "as_scalar", "as_container", "as_object", "as_union",
                                             "as_intersection"]
assert M.PropertySet is not M.AnySet is not M.ScalarSet  # each family its own

## MUT-02 · Reference verbs: an object exists, with its properties; a property changed through the nested value verbs

In [ ]:
exists = store.Mutations.Reference(lambda u: u.exists(lambda e: e.definition(lambda p: p.as_object(lambda d: (
    d
    .path(at("crm.Ada"))
    .schema("crm.Person")
    .properties({"name": "Ada", "age": 36})
)))))
aged = store.Mutations.Reference(lambda u: u.as_property(lambda p: (
    p
    .ref(ref("crm.Ada"))
    .name("age")
    .mutations([store.Mutations.OfProperty(lambda v: v.set(lambda s: s.value(37)))])
)))
deep = store.Mutations.Reference(lambda u: u.as_property(lambda p: (
    p
    .ref(ref("crm.Ada"))
    .name("address")
    .mutations([store.Mutations.OfProperty(lambda v: v.as_any(lambda a: a.mutations([
        store.Mutations.OfAny(lambda x: x.as_object(lambda o: o.mutations([
            store.Mutations.OfObject.AsProperty(lambda q: q.name("city").mutations([
                store.Mutations.OfProperty(lambda w: w.set(lambda s: s.value("Paris"))),
            ])),
        ]))),
    ])))])
)))
written = plain(log(exists, aged, deep))
assert written[0] == {"exists": {"definition": {"path": written_at("crm.Ada"), "schema": "crm.Person",
                                                "properties": {"name": {"str": "Ada"}, "age": {"int": 36}}}}}
assert written[1] == {"as_property": {"ref": {"path": written_at("crm.Ada")}, "name": "age",
                                      "mutations": [{"set": {"value": {"int": 37}}}]}}
nested = written[2]["as_property"]["mutations"][0]["as_any"]["mutations"][0]["as_object"]["mutations"][0]
assert nested == {"name": "city", "mutations": [{"set": {"value": {"str": "Paris"}}}]}  # no ref: the path gives it

## MUT-03 · Value verbs: containers by slices, by count, and item by item in modulo order; scalars; unions and intersections

In [ ]:
items = store.Mutations.OfContainer(lambda c: c.as_items(lambda i: i.mutations([
    store.Mutations.OfAny(lambda a: a.as_scalar(lambda s: s.mutations([
        store.Mutations.OfScalar(lambda x: x.set(lambda v: v.value(0))),
    ]))),
    store.Mutations.OfAny(lambda a: a.clear(store.Mutations.OfAny.Clear(lambda c: c))),
])))
edits = [
    store.Mutations.OfContainer(lambda c: c.insert(lambda i: i.slice(lambda p: p.as_object(lambda s: s.index(0).items(["a", "b"]))))),
    store.Mutations.OfContainer(lambda c: c.remove(lambda r: r.index(1).count(2))),
    store.Mutations.OfContainer(lambda c: c.append(lambda a: a.items(["z"]))),
    store.Mutations.OfContainer(lambda c: c.prepend(lambda a: a.items(["y"]))),
    store.Mutations.OfContainer(lambda c: c.clear(store.Mutations.OfContainer.Clear(lambda c: c))),
    items,
]
held = [plain(edit, M.OfContainer) for edit in edits]
assert [list(h) for h in held] == [["insert"], ["remove"], ["append"], ["prepend"], ["clear"], ["as_items"]]
assert held[1] == {"remove": {"index": {"int": 1}, "count": 2}} and held[4] == {"clear": {}}
assert held[5]["as_items"]["mutations"][1] == {"clear": {}}
branch = store.Mutations.OfAny(lambda a: a.as_union(lambda u: u.mutations([
    store.Mutations.OfUnion.AsBranch(lambda b: b.name("circle").mutations([
        store.Mutations.OfProperty(lambda p: p.clear(store.Mutations.OfProperty.Clear(lambda c: c))),
    ])),
])))
part = store.Mutations.OfAny(lambda a: a.as_intersection(lambda i: i.mutations([
    store.Mutations.OfIntersection.AsPart(lambda b: b.name("stamp").mutations([])),
])))
assert plain(branch, M.OfAny) == {"as_union": {"mutations": [{"name": "circle", "mutations": [{"clear": {}}]}]}}
assert plain(part, M.OfAny) == {"as_intersection": {"mutations": [{"name": "stamp", "mutations": []}]}}

## MUT-04 · Relations: entries inserted and removed with their links and properties; an adjacency cleared; entries in modulo order

In [ ]:
def entry(knower, known, since):
    return store.Mutations.OfEntry(lambda e: e.mutations([
        store.Mutations.OfEntry.Member(lambda m: m.as_link(lambda l: l.name("knower").mutations([
            store.Mutations.OfLink(lambda k: k.set(lambda s: s.ref(ref(knower)))),
        ]))),
        store.Mutations.OfEntry.Member(lambda m: m.as_link(lambda l: l.name("known").mutations([
            store.Mutations.OfLink(lambda k: k.set(lambda s: s.ref(ref(known)))),
        ]))),
        store.Mutations.OfEntry.Member(lambda m: m.as_property(lambda p: p.name("since").mutations([
            store.Mutations.OfProperty(lambda v: v.set(lambda s: s.value(since))),
        ]))),
    ]))


inserted = store.Mutations.Reference(lambda u: u.insert(lambda i: (
    i
    .relation(ref(".", "crm", "Knows"))
    .entries([entry("crm.Ada", "crm.Bob", 2020)])
)))
removed = store.Mutations.Reference(lambda u: u.remove(lambda r: (
    r
    .relation(ref(".", "crm", "Knows"))
    .entries([entry("crm.Ada", "crm.Bob", 2020)])
)))
cleared = store.Mutations.Reference(lambda u: u.clear(lambda c: c.ref(ref("crm.Ada")).adjacency("acquaintances")))
renumbered = store.Mutations.Reference(lambda u: u.as_entries(lambda a: (
    a
    .ref(ref("crm.Ada"))
    .adjacency("acquaintances")
    .mutations([store.Mutations.OfEntry.AsProperty(lambda p: p.name("since").mutations([
        store.Mutations.OfProperty(lambda v: v.set(lambda s: s.value(2021))),
    ]))])
)))
unlinked = store.Mutations.OfLink(lambda k: k.clear(store.Mutations.OfLink.Clear(lambda c: c)))
written = plain(log(inserted, removed, cleared, renumbered))
assert [list(w) for w in written] == [["insert"], ["remove"], ["clear"], ["as_entries"]]
first = written[0]["insert"]["entries"][0]["mutations"]
assert first[0] == {"as_link": {"name": "knower", "mutations": [{"set": {"ref": {"path": written_at("crm.Ada")}}}]}}
assert first[2] == {"as_property": {"name": "since", "mutations": [{"set": {"value": {"int": 2020}}}]}}
assert written[2] == {"clear": {"ref": {"path": written_at("crm.Ada")}, "adjacency": "acquaintances"}}
assert plain(unlinked, M.OfLink) == {"clear": {}}

## MUT-05 · A log read back from plain data and from JSON, as any value is

In [ ]:
exists = store.Mutations.Reference(lambda u: u.exists(lambda e: e.definition(lambda p: p.as_object(lambda d: (
    d
    .path(at("crm.Ada"))
    .schema("crm.Person")
    .properties({"name": "Ada"})
)))))
written = plain(log(exists))
back = Plain.FromPlain(store)(M.Log, written)
assert plain(back) == written
text = JSON.ToJSON(store)(M.Log, back)
assert json.loads(text) == written and plain(JSON.FromJSON(store)(M.Log, text)) == written
with raises(Exception):
    Plain.FromPlain(store)(M.Log, [{"exists": {}, "clear": {}}])  # a union value holds one branch